# Polymarket BTC 5M Bot - Colab Runner\n\nNotebook ini menjalankan bot dalam mode **PAPER_TRADING** untuk uji edge/winrate. Jangan isi private key di notebook ini kecuali kamu benar-benar paham risikonya. Default di bawah memaksa `REAL_TRADING=false`.

## 1. Upload ZIP Project\n\nUpload file `polymarket_btc_5m_bot_colab.zip` dari komputer lokal ke Colab.

In [ ]:
from google.colab import files\nuploaded = files.upload()\nlist(uploaded.keys())

In [ ]:
import os, zipfile, shutil\n\nzip_name = next(name for name in uploaded if name.endswith('.zip'))\nworkdir = '/content/polymarket_btc_5m_bot'\nif os.path.exists(workdir):\n    shutil.rmtree(workdir)\nwith zipfile.ZipFile(zip_name, 'r') as zf:\n    zf.extractall('/content')\n\n# Handle either zipped folder or zipped contents.\nif not os.path.exists(os.path.join(workdir, 'main.py')):\n    candidates = []\n    for root, dirs, files_ in os.walk('/content'):\n        if 'main.py' in files_ and 'risk_manager.py' in files_:\n            candidates.append(root)\n    if candidates:\n        workdir = candidates[0]\n\nos.chdir(workdir)\nprint('Working directory:', os.getcwd())\n!ls -la

## 2. Install Dependencies

In [ ]:
!python --version\n!pip install -q python-dotenv requests httpx websockets numpy pandas streamlit pytest\n# Optional: official Polymarket SDK, may change package name/version over time.\n!pip install -q py-clob-client-v2 || true

## 3. Create Safe `.env`\n\nCell ini sengaja memaksa paper trading. Untuk forward-test 24 jam, tidak perlu credential Polymarket.

In [ ]:
env_text = '''\nREAL_TRADING=false\nPAPER_STARTING_BALANCE=1000\nMAX_POSITION_PCT=0.10\nMAX_DAILY_LOSS_PCT=0.20\nEDGE_HIGH=0.16\nMIN_CONFIDENCE_HIGH=85\nMARKET_URL=https://polymarket.com/id/event/btc-updown-5m-1778690700\nDATABASE_URL=sqlite:///bot.sqlite3\nMAX_SPREAD=0.04\nMAX_SLIPPAGE=0.02\nMIN_LIQUIDITY_MULTIPLE=3\nMAX_FEED_DEVIATION_BPS=20\nPOLL_SECONDS=2\n'''\nwith open('.env', 'w', encoding='utf-8') as f:\n    f.write(env_text.strip() + '\\n')\nprint(open('.env', encoding='utf-8').read())

## 4. Run Tests

In [ ]:
!python -m pytest -q

## 5. Forward-Test Paper 24 Jam\n\nColab harus tetap hidup selama cell ini berjalan. Untuk tes cepat, ubah `--hours 24` menjadi `--hours 0.25`.

In [ ]:
!python backtester.py forward --hours 24

## 6. Report Edge / Winrate

In [ ]:
!python backtester.py report --hours 24

## 7. Optional Dashboard via Streamlit Tunnel

In [ ]:
!pip install -q streamlit\n!npm install -g localtunnel > /dev/null 2>&1\nget_ipython().system_raw('streamlit run dashboard.py --server.port 8501 --server.headless true > /content/streamlit.log 2>&1 &')\nprint('Opening tunnel. If it asks for password, use the external IP printed below.')\n!curl -s https://loca.lt/mytunnelpassword\n!npx localtunnel --port 8501

## Download Database Setelah Test

In [ ]:
from google.colab import files\nfiles.download('bot.sqlite3')